# Hero Analysis Notebook

Verifying the flattened Hero tables using rich visualization.

In [1]:
import os
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from super.core.runtime import bootstrap_spark_env
from super.core import utils
from super.core.display import display_scrollable_dataframe
# Import shared warehouse logic
from super.core.warehouse import get_hero_genome_summary, get_all_heroes_data


In [2]:

# Bootstrap & Session
bootstrap_spark_env()
spark = SparkSession.builder.appName("HeroAnalyzer").getOrCreate()

# Config
conf = utils.get_app_conf("generate_powers")
warehouse_root = os.path.join(conf.get_string("stage_root"), "warehouse")
print(f"Reading Warehouse: {warehouse_root}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/19 12:15:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Reading Warehouse: /home/gideon/tmp/super_powers/data/warehouse


## 1. Hero Genome Reports (Unified View - Spark)
Analysis of the **Gene Cluster** for each hero using Spark.

In [3]:
# Use shared logic for DRY compliance
full_report = get_hero_genome_summary(spark, warehouse_root)

# Display as a Form-like table
display_scrollable_dataframe(full_report.orderBy("hero_name").toPandas())

## 2. Deep Dive: Bugs Bunny's Cluster (Spark)

In [4]:
def get_hero_report(_full_report, hero_name, print_report=True):
    hero_report = _full_report.filter(F.col("hero_name") == hero_name)

    # Show full text for Bugs to verify richness
    for row in hero_report.collect():

        # create rich text for printing with line breaks
        report = f"""
            "hero_name": {row.hero_name},
            "ontology": {row.ontology},
            "bio": {row.bio},
            "master_regulator_id": {row.master_regulator_id},
            "mutation_class": {row.mutation_class},
            "cluster_size": {row.cluster_size},
            "cluster_network_summary": {row.cluster_network_summary},
        """

        if print_report:
            print(report)

    return report


In [ ]:


a_report = get_hero_report(_full_report=full_report, hero_name="High Lander")



            "hero_name": Bob Sponge,
            "ontology": generated,
            "bio": A hero with the strength of a titan.,
            "master_regulator_id": MIGHTY-001,
            "mutation_class": Canonical,
            "cluster_size": 15,
            "cluster_network_summary": LINK-0(Boost:0.50), LINK-1(Boost:0.50), LINK-2(Boost:0.50), LINK-3(Boost:0.50), LINK-4(Boost:0.50), LINK-5(Boost:0.50), LINK-6(Boost:0.50), LINK-7(Boost:0.50), LINK-8(Boost:0.50), LINK-9(Boost:0.50), LINK-10(Boost:0.50), LINK-11(Boost:0.50), LINK-12(Boost:0.50), LINK-13(Boost:0.50), LINK-14(Boost:0.50),
        


## 3. Interactive Ontology View (Pandas)
View the hero table filtered by ontology using **Pandas** directly.

In [ ]:
# 1. List Available Ontologies
profiles_path = os.path.join(warehouse_root, "hero_profiles")
partitions = [d for d in os.listdir(profiles_path) if d.startswith("ontology=")]
print("Available Ontologies:")
for p in partitions:
    print(f" - {p.replace('ontology=', '')}")

In [ ]:
# 2. Configure Ontology
# CHANGE THIS VALUE to filter by a different universe (e.g. "Marvel", "DC Comics", "Looney Tunes")
SELECTED_ONTOLOGY = "Looney Tunes"

print(f"Loading data for: {SELECTED_ONTOLOGY}...")
pandas_report = get_all_heroes_data(warehouse_root=warehouse_root, ontology=SELECTED_ONTOLOGY)

display_scrollable_dataframe(pandas_report)

## 4. Data Integrity Checks
Finding **Ontologyless** (missing universe) and **Genomeless** (failed generation) heroes.

In [ ]:
import pandas as pd

# 1. Ontologyless Heroes
print("Checking for Ontologyless Heroes...")
try:
    p_path = os.path.join(warehouse_root, "hero_profiles")
    df_p = pd.read_parquet(p_path, engine='pyarrow')
    
    ontologyless = df_p[df_p['ontology'].isna() | (df_p['ontology'] == '')]
    
    if not ontologyless.empty:
        print(f"Found {len(ontologyless)} heroes without ontology!")
        display_scrollable_dataframe(ontologyless)
    else:
        print(f"✅ All {len(df_p)} heroes have an assigned ontology.")
        
except Exception as e:
    print(f"Error checking profiles: {e}")

# 2. Genomeless Heroes
print("\nChecking for Genomeless Heroes (Profile exists, but Genes missing)...")
try:
    g_path = os.path.join(warehouse_root, "hero_genes")
    df_g = pd.read_parquet(g_path, engine='pyarrow')
    
    # Left join Profile -> Gene on Name AND Ontology (Partition)
    # We need to ensure we join on keys that exist in both
    merged = pd.merge(df_p, df_g, on=["hero_name", "ontology"], how="left", indicator=True, suffixes=('', '_gene'))
    
    # Find rows where merge failed (left_only)
    # We filter for uniqueness on hero_name to report list of heroes
    genomeless = merged[merged['_merge'] == 'left_only'].drop_duplicates(subset=['hero_name'])
    
    if not genomeless.empty:
        print(f"Found {len(genomeless)} heroes without generated genetics!")
        cols = ['hero_name', 'ontology', 'bio']
        display_scrollable_dataframe(genomeless[[c for c in cols if c in genomeless.columns]])
    else:
        print("✅ All heroes have genetic data.")
        
except Exception as e:
    print(f"Error checking genes: {e}")